In [1]:
import pandas as pd
import numpy as np
import dask.dataframe as dd
import dask
import glob
import pickle
import os
import numpy as np
from scipy.optimize import brentq
from scipy.stats import norm
from datetime import datetime, time
from pathlib import Path
from typing import Dict, List, Tuple

In [2]:
def load_data(
    csv_path: str,
    save_parquet: bool = False,
    parquet_path: str | None = None,
    preview: bool = True,
    preview_rows: int = 5
) -> pd.DataFrame:
    """
    Load spot CSV with 'Date' and 'Time' columns, create a 'Datetime' column,
    drop unparseable rows, and (optionally) save & reload via Parquet.

    Parameters
    ----------
    csv_path : str
        Path to the input CSV. Must contain columns:
        ['Date', 'Time', 'Open', 'High', 'Low', 'Close'].
    save_parquet : bool, optional
        If True, save the cleaned DataFrame to Parquet and reload it.
    parquet_path : str | None, optional
        Destination Parquet path. If None and save_parquet=True,
        uses csv_path with '.parquet' extension.
    preview : bool, optional
        If True, prints a small preview of the DataFrame.
    preview_rows : int, optional
        Number of rows to show in the preview.

    Returns
    -------
    pd.DataFrame
        DataFrame with columns ['Datetime', 'Open', 'High', 'Low', 'Close'].
    """
    # Load CSV with required columns
    spot_df = pd.read_csv(
        csv_path,
        usecols=["Date", "Time", "Open", "High", "Low", "Close"]
    )

    # Build Datetime from Date + Time
    spot_df["Datetime"] = pd.to_datetime(
        spot_df["Date"].astype(str) + " " + spot_df["Time"].astype(str),
        errors="coerce"
    )

    # Drop rows with bad datetime parsing
    num_bad_spot = spot_df["Datetime"].isna().sum()
    if num_bad_spot > 0:
        print(f"⚠️  Dropping {num_bad_spot} bad rows from spot_df due to unparseable Datetime.")
        spot_df = spot_df.dropna(subset=["Datetime"])

    # Keep the required columns
    spot_df = spot_df[["Datetime", "Open", "High", "Low", "Close"]]

    # Optional preview
    if preview:
        print("spot_df preview:")
        print(spot_df.head(preview_rows))

    # Optionally save & reload via Parquet
    if save_parquet:
        try:
            if parquet_path is None:
                parquet_path = str(Path(csv_path).with_suffix(".parquet"))
            spot_df.to_parquet(parquet_path, index=False)
            # Reload to verify and standardize downstream usage
            spot_df = pd.read_parquet(parquet_path)
            print("spot_df loaded successfully from parquet.")
        except Exception as e:
            print(f"Error saving/loading spot_df: {e}")

    return spot_df


In [3]:
def make_spot_60min(
    spot_df: pd.DataFrame,
    start_time: str = "09:15:00",
    end_time: str = "15:15:00",
    resample_rule: str = "60min",
    offset: str = "15min",
    ema_span: int = 200,
    preview: bool = False,
    preview_rows: int = 5,
) -> pd.DataFrame:
    """
    Convert tick/1-min spot data into 60-min OHLC bars aligned to NIFTY hours
    and compute EMA_200.

    Parameters
    ----------
    spot_df : pd.DataFrame
        Must have columns: ['Datetime', 'Open', 'High', 'Low', 'Close'].
    start_time : str
        Trading start time for between_time filter (e.g., '09:15:00').
    end_time : str
        Trading end time for between_time filter (e.g., '15:15:00').
    resample_rule : str
        Pandas resample rule for bar size (e.g., '60min').
    offset : str
        Offset from midnight for aligning first bar (e.g., '15min' for 9:15 open).
    ema_span : int
        EMA period applied on Close.
    preview : bool
        Print tail of the resulting DataFrame if True.
    preview_rows : int
        Number of rows to print in the preview.

    Returns
    -------
    pd.DataFrame
        Columns: ['Datetime','Open','High','Low','Close','EMA_200'] with 60-min bars.
    """
    required_cols = {"Datetime", "Open", "High", "Low", "Close"}
    missing = required_cols.difference(spot_df.columns)
    if missing:
        raise ValueError(f"spot_df is missing required columns: {sorted(missing)}")

    # Ensure Datetime is datetime and sorted
    df = spot_df.copy()
    df["Datetime"] = pd.to_datetime(df["Datetime"], errors="coerce")
    df = df.dropna(subset=["Datetime"]).sort_values("Datetime")

    # Index for resampling
    df = df.set_index("Datetime")

    # Filter regular trading hours
    df = df.between_time(start_time, end_time)

    # Resample to OHLC (no volume provided)
    spot_60min = df.resample(
        resample_rule,
        origin="start_day",
        offset=offset,
        label="left",
        closed="left",
    ).agg({
        "Open": "first",
        "High": "max",
        "Low": "min",
        "Close": "last",
    }).dropna()

    # Back to columns
    spot_60min = spot_60min.reset_index()

    # EMA 200 on Close
    spot_60min["EMA_200"] = spot_60min["Close"].ewm(span=ema_span, adjust=False).mean()

    if preview:
        print(spot_60min.tail(preview_rows))

    return spot_60min





In [4]:
def og_signals_generator(spot_60min: pd.DataFrame, preview: bool = False) -> pd.DataFrame:
    """
    Generate OG 7-bar breakout signals with EMA-200 entry filter.
    Returns a DataFrame with columns ['Datetime', 'Signal', 'Direction'].

    Requirements in spot_60min:
        Columns: ['Datetime','High','Low','Close','EMA_200']
    """
    required = {"Datetime", "High", "Low", "Close", "EMA_200"}
    missing = required.difference(spot_60min.columns)
    if missing:
        raise ValueError(f"spot_60min missing required columns: {sorted(missing)}")

    df = spot_60min.sort_values('Datetime').reset_index(drop=True).copy()

    # Pre-calculate rolling indicators (lagged so current bar can't see itself)
    df['7bar_High'] = df['High'].shift(1).rolling(window=7).max()
    df['7bar_Low'] = df['Low'].shift(1).rolling(window=7).min()
    df['7bar_Close_High'] = df['Close'].shift(1).rolling(window=7).max()
    df['7bar_Close_Low'] = df['Close'].shift(1).rolling(window=7).min()

    signals = []
    in_position = None   # None, "Bullish", "Bearish"

    for i in range(len(df) - 1):
        row = df.iloc[i]
        next_row = df.iloc[i + 1]
        dt = next_row['Datetime']
        ema = row['EMA_200']
        close = row['Close']
        low = row['Low']
        high = row['High']

        # --- When flat, decide on first entry side using EMA 200 ---
        if in_position is None:
            if close > ema:
                # Only consider Bullish_Entry, using EMA filter
                entry_cond = (low < row['7bar_Low']) and (close < row['7bar_Close_Low'])
                if entry_cond:
                    signals.append({'Datetime': dt, 'Signal': 'Bullish_Entry', 'Direction': 'Bullish'})
                    in_position = "Bullish"
            elif close < ema:
                # Only consider Bearish_Entry, using EMA filter
                entry_cond = (high > row['7bar_High']) and (close > row['7bar_Close_High'])
                if entry_cond:
                    signals.append({'Datetime': dt, 'Signal': 'Bearish_Entry', 'Direction': 'Bearish'})
                    in_position = "Bearish"
            # If close == ema or no entry, remain flat

        # --- When already in a trade (re-entries & exits ignore EMA filter) ---
        elif in_position == "Bullish":
            # Bullish Reentry/Exit (IGNORE EMA)
            entry_cond = (low < row['7bar_Low']) and (close < row['7bar_Close_Low'])
            exit_cond = (high > row['7bar_High']) and (close > row['7bar_Close_High'])
            if entry_cond:
                signals.append({'Datetime': dt, 'Signal': 'Bullish_Reentry', 'Direction': 'Bullish'})
            elif exit_cond:
                signals.append({'Datetime': dt, 'Signal': 'Bullish_Exit', 'Direction': 'Bullish'})
                in_position = None

        elif in_position == "Bearish":
            # Bearish Reentry/Exit (IGNORE EMA)
            entry_cond = (high > row['7bar_High']) and (close > row['7bar_Close_High'])
            exit_cond = (low < row['7bar_Low']) and (close < row['7bar_Close_Low'])
            if entry_cond:
                signals.append({'Datetime': dt, 'Signal': 'Bearish_Reentry', 'Direction': 'Bearish'})
            elif exit_cond:
                signals.append({'Datetime': dt, 'Signal': 'Bearish_Exit', 'Direction': 'Bearish'})
                in_position = None

    signals_df = pd.DataFrame(signals, columns=['Datetime', 'Signal', 'Direction']) if signals else \
                 pd.DataFrame(columns=['Datetime', 'Signal', 'Direction'])

    if preview:
        print(signals_df.head(20))
        print(f"Total signals generated: {len(signals_df)}")

    return signals_df

In [5]:
def making_tradelogs(
    spot_60min: pd.DataFrame,
    signals_df: pd.DataFrame,
    preview: bool = False
) -> pd.DataFrame:
    """
    Build a trade log from OG signals and 60-min spot bars.
      - Entries/Reentries open a new trade at the signal bar's OPEN price
      - Exits close ALL open trades on that side at the signal bar's OPEN price
      - Any leftover open trades at the end are force-closed at the last bar's OPEN
      - Duplicate rows are dropped at the end and index is reset

    Parameters
    ----------
    spot_60min : pd.DataFrame
        Must include: ['Datetime','Open','Close'].
    signals_df : pd.DataFrame
        Must include: ['Datetime','Signal','Direction'].
    preview : bool
        If True, prints tail and summary stats.

    Returns
    -------
    pd.DataFrame
        Columns: ['Entry Time','Exit Time','Side','Entry Price','Exit Price','Signal Type','PnL']
    """
    # --- Validate inputs ---
    req_spot = {"Datetime", "Open", "Close"}
    req_sig = {"Datetime", "Signal", "Direction"}
    miss_spot = req_spot.difference(spot_60min.columns)
    miss_sig = req_sig.difference(signals_df.columns)
    if miss_spot:
        raise ValueError(f"spot_60min missing required columns: {sorted(miss_spot)}")
    if miss_sig:
        raise ValueError(f"signals_df missing required columns: {sorted(miss_sig)}")

    # --- Ensure correct types/sorting ---
    spot_60min = spot_60min.copy()
    signals_df = signals_df.copy()

    spot_60min["Datetime"] = pd.to_datetime(spot_60min["Datetime"], errors="coerce")
    signals_df["Datetime"] = pd.to_datetime(signals_df["Datetime"], errors="coerce")

    spot_60min = spot_60min.dropna(subset=["Datetime"]).sort_values("Datetime").reset_index(drop=True)
    signals_df = signals_df.dropna(subset=["Datetime"]).sort_values("Datetime").reset_index(drop=True)

    # --- Merge to get prices for signal times (we'll use Open) ---
    signals_df = pd.merge(
        signals_df,
        spot_60min[["Datetime", "Open", "Close"]],
        how="left",
        on="Datetime"
    )

    # --- Build trade log ---
    trade_log = []
    open_trades = []

    for idx, row in signals_df.iterrows():
        signal = row["Signal"]
        dt = row["Datetime"]
        price = row["Open"]  # always use OPEN for entries/exits
        direction = row["Direction"]

        # ENTRY/REENTRY: add new open trade
        if ("Entry" in signal) or ("Reentry" in signal):
            open_trades.append({
                "Entry Time": dt,
                "Entry Price": price,
                "Side": direction,
                "Signal Type": signal,
                "Signal Index": idx
            })

        # EXIT: close all open trades on this side
        elif "Exit" in signal:
            to_close = [t for t in open_trades if t["Side"] == direction]
            for trade in to_close:
                trade["Exit Time"] = dt
                trade["Exit Price"] = price
                trade["Exit Signal Index"] = idx
                # PnL: Long = Exit - Entry, Short = Entry - Exit
                if direction == "Bullish":
                    trade["PnL"] = trade["Exit Price"] - trade["Entry Price"]
                else:
                    trade["PnL"] = trade["Entry Price"] - trade["Exit Price"]
                trade_log.append(trade)
            # remove closed trades from open_trades
            open_trades = [t for t in open_trades if t["Side"] != direction]

    # --- Force-close any remaining open positions at the last available OPEN ---
    if open_trades:
        final_price = spot_60min.iloc[-1]["Open"]
        final_time = spot_60min.iloc[-1]["Datetime"]
        for trade in open_trades:
            trade["Exit Time"] = final_time
            trade["Exit Price"] = final_price
            if trade["Side"] == "Bullish":
                trade["PnL"] = final_price - trade["Entry Price"]
            else:
                trade["PnL"] = trade["Entry Price"] - final_price
            trade["Forced Exit"] = True
            trade_log.append(trade)

    # --- DataFrame & clean columns ---
    trade_log_df = pd.DataFrame(trade_log)
    if trade_log_df.empty:
        trade_log_df = pd.DataFrame(columns=[
            "Entry Time", "Exit Time", "Side",
            "Entry Price", "Exit Price", "Signal Type", "PnL"
        ])
    else:
        trade_log_df = trade_log_df[
            ["Entry Time", "Exit Time", "Side", "Entry Price", "Exit Price", "Signal Type", "PnL"]
        ]

    # --- NEW: Drop duplicate rows and reset index ---
    trade_log_df = trade_log_df.drop_duplicates().reset_index(drop=True)

    # --- Optional preview ---
    if preview:
        print(trade_log_df.tail(20))
        total = len(trade_log_df)
        win_rate = (trade_log_df["PnL"] > 0).mean() * 100 if total else 0.0
        avg_pnl = trade_log_df["PnL"].mean() if total else 0.0
        print(f"Total trades: {total}")
        print(f"Win rate: {win_rate:.2f}% | Avg PnL: {avg_pnl:.2f}")

    return trade_log_df

In [6]:
def loading_option_data(
    years_to_load: List[int],
    outdir: str = "./options_data_yearwise",
    nearest_glob_template: str = "/home/newberry3/main/Data/NIFTY/NIFTY_{year}*.pkl",
    next_glob_template: str = "/home/newberry3/main/Data/2ndweeknext_Expiry/NIFTY_{year}*.pkl",
    save_merged: bool = True,
    assign_globals: bool = True,
) -> Tuple[Dict[int, pd.DataFrame], Dict[int, pd.DataFrame], Dict[int, pd.DataFrame]]:
    """
    Load option data year-wise from PKL files (nearest & next expiry), normalize columns,
    create DateTime, save to Parquet per year, then reload and build merged DataFrames.

    Returns (nearest_dict, next_dict, merged_dict).
    Also (optionally) assigns globals:
      options_df_<year>, options_df_2_<year>, options_df_merged_<year>
    """
    os.makedirs(outdir, exist_ok=True)

    # ---------- Phase 1: Load PKLs year-wise, normalize, save to Parquet ----------
    for year in years_to_load:
        print(f"\n--- Loading option data for year: {year} ---")

        # ===== NEAREST EXPIRY =====
        options_files = glob.glob(nearest_glob_template.format(year=year))
        if not options_files:
            print(f"No .pkl files found in {nearest_glob_template.format(year=year)}")
        else:
            # Collect union of columns
            all_cols = set()
            for file in options_files:
                try:
                    with open(file, "rb") as f:
                        df = pickle.load(f)
                except Exception as e:
                    print(f"  ⚠️  Skip (read error): {file} -> {e}")
                    continue
                df = df.drop(columns=[c for c in ['OI', 'Volume'] if c in df.columns], errors='ignore')
                all_cols.update(df.columns)
            all_cols = sorted(list(all_cols))

            dfs = []
            for file in options_files:
                try:
                    with open(file, "rb") as f:
                        df = pickle.load(f)
                except Exception as e:
                    print(f"  ⚠️  Skip (read error): {file} -> {e}")
                    continue
                df = df.drop(columns=[c for c in ['OI', 'Volume'] if c in df.columns], errors='ignore')
                # Fill missing columns
                for c in all_cols:
                    if c not in df.columns:
                        df[c] = pd.NA
                df = df[all_cols]
                dfs.append(df)

            if dfs:
                options_df = pd.concat(dfs, ignore_index=True)

                # Stringify core cols (if present)
                for c in ['StrikePrice', 'ExpiryDate', 'Ticker', 'Date', 'Time', 'Type']:
                    if c in options_df.columns:
                        options_df[c] = options_df[c].astype(str)

                # Build DateTime
                if {'Date', 'Time'}.issubset(options_df.columns):
                    options_df['DateTime'] = pd.to_datetime(
                        options_df['Date'] + " " + options_df['Time'], errors='coerce'
                    )
                elif 'DateTime' in options_df.columns:
                    options_df['DateTime'] = pd.to_datetime(options_df['DateTime'], errors='coerce')
                else:
                    raise ValueError("Nearest: Could not construct DateTime (no Date/Time or DateTime present).")

                options_df = options_df.dropna(subset=['DateTime'])

                # Clean columns & types
                cols_to_drop = ['Ticker', 'High', 'Low', 'Close', 'Date', 'Time']
                options_df = options_df.drop(columns=[c for c in cols_to_drop if c in options_df.columns], errors='ignore')

                # Ensure ordering with DateTime first
                cols = ['DateTime'] + [c for c in options_df.columns if c != 'DateTime']
                options_df = options_df[cols]

                if 'StrikePrice' in options_df.columns:
                    options_df['StrikePrice'] = pd.to_numeric(options_df['StrikePrice'], errors='coerce')
                if 'Type' in options_df.columns:
                    options_df['Type'] = options_df['Type'].astype(str).str.strip().str.upper()
                if 'ExpiryDate' in options_df.columns:
                    options_df['ExpiryDate'] = pd.to_datetime(options_df['ExpiryDate'], errors='coerce').dt.normalize()

                # Save
                nearest_path = os.path.join(outdir, f"options_nearest_{year}.parquet")
                options_df.to_parquet(nearest_path, index=False)
                print(f"✅ Saved NEAREST expiry options for {year} to {nearest_path}")
            else:
                print(f"  ⚠️  No readable PKLs for nearest expiry in {year}")

        # ===== NEXT EXPIRY =====
        options_files_2 = glob.glob(next_glob_template.format(year=year))
        if not options_files_2:
            print(f"No .pkl files found in {next_glob_template.format(year=year)}")
        else:
            all_cols_2 = set()
            # First pass: union columns
            for file in options_files_2:
                try:
                    with open(file, "rb") as f:
                        df2 = pickle.load(f)
                except Exception as e:
                    print(f"  ⚠️  Skip (read error): {file} -> {e}")
                    continue
                df2 = df2.drop(columns=[c for c in ['OI', 'Volume'] if c in df2.columns], errors='ignore')
                all_cols_2.update(df2.columns)
            all_cols_2 = sorted(list(all_cols_2))

            dfs_2 = []
            for file in options_files_2:
                try:
                    with open(file, "rb") as f:
                        df2 = pickle.load(f)
                except Exception as e:
                    print(f"  ⚠️  Skip (read error): {file} -> {e}")
                    continue
                df2 = df2.drop(columns=[c for c in ['OI', 'Volume'] if c in df2.columns], errors='ignore')
                for c in all_cols_2:
                    if c not in df2.columns:
                        df2[c] = pd.NA
                df2 = df2[all_cols_2]

                # Drop NA in key cols if present
                key_cols = ['StrikePrice', 'ExpiryDate', 'Date', 'Time', 'Type']
                keep_cols = [c for c in key_cols if c in df2.columns]
                if keep_cols:
                    df2 = df2.dropna(subset=keep_cols)

                # Build DateTime
                if {'Date', 'Time'}.issubset(df2.columns):
                    dt_strings = df2['Date'].astype(str) + ' ' + df2['Time'].astype(str)
                    df2['DateTime'] = pd.to_datetime(dt_strings, errors='coerce')
                elif 'DateTime' in df2.columns:
                    df2['DateTime'] = pd.to_datetime(df2['DateTime'], errors='coerce')
                else:
                    raise ValueError("Next: Could not construct DateTime (no Date/Time or DateTime present).")
                df2 = df2.dropna(subset=['DateTime'])

                dfs_2.append(df2)

            if dfs_2:
                options_df_2 = pd.concat(dfs_2, ignore_index=True)

                for c in ['StrikePrice', 'ExpiryDate', 'Ticker', 'Date', 'Time', 'Type']:
                    if c in options_df_2.columns:
                        options_df_2[c] = options_df_2[c].astype(str)
                if 'StrikePrice' in options_df_2.columns:
                    options_df_2['StrikePrice'] = pd.to_numeric(options_df_2['StrikePrice'], errors='coerce')
                if 'Type' in options_df_2.columns:
                    options_df_2['Type'] = options_df_2['Type'].astype(str).str.strip().str.upper()
                if 'ExpiryDate' in options_df_2.columns:
                    options_df_2['ExpiryDate'] = pd.to_datetime(options_df_2['ExpiryDate'], errors='coerce').dt.normalize()

                cols_to_drop = ['Ticker', 'High', 'Low', 'Close', 'Date', 'Time']
                options_df_2 = options_df_2.drop(columns=[c for c in cols_to_drop if c in options_df_2.columns], errors='ignore')

                cols = ['DateTime'] + [c for c in options_df_2.columns if c != 'DateTime']
                options_df_2 = options_df_2[cols]

                # Save
                next_path = os.path.join(outdir, f"options_next_{year}.parquet")
                options_df_2.to_parquet(next_path, index=False)
                print(f"✅ Saved NEXT expiry options for {year} to {next_path}")
            else:
                print(f"  ⚠️  No readable PKLs for next expiry in {year}")

    print(f"\nAll options data saved to {outdir}")

    # ---------- Phase 2: Reload Parquets, create dicts & merged ----------
    nearest_dict: Dict[int, pd.DataFrame] = {}
    next_dict: Dict[int, pd.DataFrame] = {}
    merged_dict: Dict[int, pd.DataFrame] = {}

    for year in years_to_load:
        nearest_path = os.path.join(outdir, f"options_nearest_{year}.parquet")
        next_path = os.path.join(outdir, f"options_next_{year}.parquet")

        # Load if both present
        try:
            options_df = pd.read_parquet(nearest_path)
            options_df_2 = pd.read_parquet(next_path)
            print(f"Loaded: {nearest_path} and {next_path}")
        except Exception as e:
            print(f"Skipping year {year} due to load error: {e}")
            continue

        nearest_dict[year] = options_df
        next_dict[year] = options_df_2

        # Merge
        merged = pd.concat([options_df, options_df_2], ignore_index=True)
        merged = merged.drop_duplicates(
            subset=['DateTime', 'ExpiryDate', 'StrikePrice', 'Type'], keep='first'
        ).reset_index(drop=True)
        merged = merged.sort_values(['DateTime', 'ExpiryDate', 'StrikePrice', 'Type'])

        merged_dict[year] = merged

        if save_merged:
            merged_path = os.path.join(outdir, f"options_df_merged_{year}.parquet")
            merged.to_parquet(merged_path, index=False)
            print(f"✅ Saved merged DataFrame for {year} to {merged_path}")

    # Optional: assign globals for convenience (mirrors your original behavior)
    if assign_globals:
        for year in years_to_load:
            if year in nearest_dict:
                globals()[f"options_df_{year}"] = nearest_dict[year]
            if year in next_dict:
                globals()[f"options_df_2_{year}"] = next_dict[year]
            if year in merged_dict:
                globals()[f"options_df_merged_{year}"] = merged_dict[year]

    return nearest_dict, next_dict, merged_dict


In [7]:
# BS/IV functions

# --- Black-Scholes Option Price ---
def black_scholes_price(S, K, T, r, sigma, option_type):
    if sigma <= 0 or T <= 0:
        return 0

    d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    if option_type == "call":
        return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    elif option_type == "put":
        return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)
    else:
        return None
    
# --- Implied Volatility from Option Price ---
def implied_volatility(option_price, S, K, T, r, option_type):
    """
    Uses Brent's method to find implied volatility from the market price.
    """
    try:
        return brentq(
            lambda sigma: black_scholes_price(S, K, T, r, sigma, option_type) - option_price,
            a=0.01,
            b=3.0,
            maxiter=1000,
            xtol=1e-6
        )
    except (ValueError, RuntimeError):
        return None
    
# --- Black-Scholes Greeks ---
def black_scholes_greeks(S, K, T, r, sigma, option_type):
    if sigma <= 0 or T <= 0:
        return None

    # Use synthetic future price
    F = S * np.exp(r * T)

    d1 = (np.log(F / K) + 0.5 * sigma ** 2 * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    if option_type == "call":
        delta = np.exp(-r * T) * norm.cdf(d1)
        theta = (-F * norm.pdf(d1) * sigma / (2 * np.sqrt(T)) - r * K * np.exp(-r * T) * norm.cdf(d2)) / 365
        rho = K * T * np.exp(-r * T) * norm.cdf(d2) / 100
    else:
        delta = -np.exp(-r * T) * norm.cdf(-d1)
        theta = (-F * norm.pdf(d1) * sigma / (2 * np.sqrt(T)) + r * K * np.exp(-r * T) * norm.cdf(-d2)) / 365
        rho = -K * T * np.exp(-r * T) * norm.cdf(-d2) / 100

    gamma = norm.pdf(d1) / (F * sigma * np.sqrt(T))
    vega = F * norm.pdf(d1) * np.sqrt(T) / 100

    return {
        'Delta': round(delta, 5),
        'Gamma': round(gamma, 5),
        'Vega': round(vega, 5),
        'Theta': round(theta, 5),
        'Rho': round(rho, 5)
    }

# --- Time to Expiry (Fractional) ---
def calculate_time_to_expiry(manual_datetime_str, expiry_date_str):
    now = datetime.strptime(manual_datetime_str, "%Y-%m-%d %H:%M:%S")
    expiry_date = datetime.strptime(expiry_date_str, "%d-%m-%y").date()

    market_open = time(9, 15)
    market_close = time(15, 30)
    today = now.date()
    days_left = (expiry_date - today).days

    if days_left <= 0:
        days_left += 1

    total_trading_minutes = (market_close.hour * 60 + market_close.minute) - (market_open.hour * 60 + market_open.minute)
    current_minutes_since_open = (now.hour * 60 + now.minute) - (market_open.hour * 60 + market_open.minute)

    if current_minutes_since_open < 0:
        T = round(days_left / 365, 6)
    elif current_minutes_since_open >= total_trading_minutes:
        T = round(max(0, (days_left - 1) / 365), 6)
    else:
        fraction_of_day_passed = current_minutes_since_open / total_trading_minutes
        T = round((days_left - fraction_of_day_passed) / 365, 6)

    print(f"\nManual Time Entered: {now}")
    print(f"Expiry Date: {expiry_date}")
    print(f"Days to Expiry (with fraction): {days_left - (fraction_of_day_passed if 0 <= current_minutes_since_open < total_trading_minutes else 0):.6f}")
    print(f"Time to Expiry in Years (T): {T}")

    return T

In [8]:
#OG+ MOD 4
def strike_entry_og(row, options_df, options_df_2, r=0.066):
   
 # --- Extract info ---
    entry_time = pd.to_datetime(row['Entry Time'])
    spot = row['Entry Price']
    side = row['Side']
    opt_type = 'PE' if side == 'Bullish' else 'CE'
    target_delta = -0.4 if side == 'Bullish' else 0.4

    # --- Pick correct DataFrame and Expiry ---
    weekday = entry_time.weekday()  # 0=Monday, ..., 4=Friday
    if weekday == 4:
        df = options_df
        expiry_list = df.loc[df['DateTime'] >= entry_time, 'ExpiryDate'].drop_duplicates().sort_values()
        expiry_source = 'NEAREST (Friday)'
    else:
        df = options_df_2
        expiry_list = df.loc[df['DateTime'] >= entry_time, 'ExpiryDate'].drop_duplicates().sort_values()
        expiry_source = 'NEXT (Mon-Thu)'

    # Ensure correct types
    df = df.copy()
    df['StrikePrice'] = df['StrikePrice'].astype(float)
    df['ExpiryDate'] = pd.to_datetime(df['ExpiryDate'])
    df['DateTime'] = pd.to_datetime(df['DateTime'])
    df['Type'] = df['Type'].astype(str)

    # Select expiry
    if expiry_list.empty:
        print(f"[DEBUG] No expiry found for {entry_time} ({side}), source={expiry_source}")
        return None
    expiry = pd.to_datetime(expiry_list.iloc[0])

    # Build time window (here, zero minutes—exact match only as in your code)
    time_window = pd.Timedelta(minutes=1)
    time_diff = (df['DateTime'] - entry_time).abs()

    # Filtering options: entry datetime, expiry, type, time within window
    mask = (
        (df['DateTime'].dt.floor('min') == entry_time.floor('min')) &  # For minute-level exact match
        (df['ExpiryDate'] == expiry) &
        (df['Type'] == opt_type) &
        (time_diff <= time_window)
    )
    df_opts = df[mask].copy()
    print(f"[DEBUG] Entry: {entry_time}, Expiry used: {expiry.date()}, "
          f"Type: {opt_type}, Rows in window (±0min): {len(df_opts)}, Source: {expiry_source}")

    if df_opts.empty:
        print(f"[DEBUG] NO OPTION DATA: {entry_time}, Expiry: {expiry.date()}, "
              f"Type: {opt_type}, Data source: {expiry_source}")
        return None

    best_row = None
    best_delta_diff = np.inf

    for idx, opt in df_opts.iterrows():
        K = float(opt['StrikePrice'])
        option_price = float(opt['Open'])
        T = calculate_time_to_expiry(entry_time.strftime("%Y-%m-%d %H:%M:%S"), expiry.strftime("%d-%m-%y"))
        option_type_bs = 'put' if opt_type == 'PE' else 'call'

        # --- IV calculation ---
        iv = implied_volatility(option_price, spot, K, T, r, option_type_bs)
        if iv is None or iv <= 0:
            print(f"  [DEBUG] IV fail at strike {K} (price={option_price:.2f}), T={T:.6f}")
            continue

        greeks = black_scholes_greeks(spot, K, T, r, iv, option_type_bs)
        if greeks is None:
            print(f"  [DEBUG] Greeks fail at strike {K} (IV={iv:.4f})")
            continue

        delta = greeks['Delta']
        delta_diff = abs(delta - target_delta)
        print(f"  [DEBUG] Strike {K}, Delta={delta:.5f}, IV={iv:.4f}, Δ={delta_diff:.5f}")

        if delta_diff < best_delta_diff:
            best_delta_diff = delta_diff
            best_row = {
                'Entry Time': entry_time,
                'Side': side,
                'Strike': K,
                'Expiry': expiry,
                'IV': iv,
                'Delta': delta,
                'Option Type': opt_type,
                'Option Price': option_price
            }
    if best_row is None:
        print(f"[DEBUG] No strike within delta range for {entry_time} {side}, expiry={expiry.date()}")
    return best_row

# # Usage
# results = []
# for idx, row in trade_log_df.iterrows():
#     res = find_best_strike_for_signal(row, options_df, options_df_2)
#     results.append(res)

# strikes_df = pd.DataFrame([x for x in results if x is not None])
# print(strikes_df)

def find_best_strike_for_signal(row, options_df, options_df_2, r=0.066):
   
 # --- Extract info ---
    entry_time = pd.to_datetime(row['Entry Time'])
    spot = row['Entry Price']
    side = row['Side']
    opt_type = 'PE' if side == 'Bullish' else 'CE'
    target_delta = -0.4 if side == 'Bullish' else 0.4

    # --- Pick correct DataFrame and Expiry ---
    weekday = entry_time.weekday()  # 0=Monday, ..., 4=Friday
    if weekday == 4:
        df = options_df
        expiry_list = df.loc[df['DateTime'] >= entry_time, 'ExpiryDate'].drop_duplicates().sort_values()
        expiry_source = 'NEAREST (Friday)'
    else:
        df = options_df_2
        expiry_list = df.loc[df['DateTime'] >= entry_time, 'ExpiryDate'].drop_duplicates().sort_values()
        expiry_source = 'NEXT (Mon-Thu)'

    # Ensure correct types
    df = df.copy()
    df['StrikePrice'] = df['StrikePrice'].astype(float)
    df['ExpiryDate'] = pd.to_datetime(df['ExpiryDate'])
    df['DateTime'] = pd.to_datetime(df['DateTime'])
    df['Type'] = df['Type'].astype(str)

    # Select expiry
    if expiry_list.empty:
        print(f"[DEBUG] No expiry found for {entry_time} ({side}), source={expiry_source}")
        return None
    expiry = pd.to_datetime(expiry_list.iloc[0])

    # Build time window (here, zero minutes—exact match only as in your code)
    time_window = pd.Timedelta(minutes=1)
    time_diff = (df['DateTime'] - entry_time).abs()

    # Filtering options: entry datetime, expiry, type, time within window
    mask = (
        (df['DateTime'].dt.floor('min') == entry_time.floor('min')) &  # For minute-level exact match
        (df['ExpiryDate'] == expiry) &
        (df['Type'] == opt_type) &
        (time_diff <= time_window)
    )
    df_opts = df[mask].copy()
    print(f"[DEBUG] Entry: {entry_time}, Expiry used: {expiry.date()}, "
          f"Type: {opt_type}, Rows in window (±0min): {len(df_opts)}, Source: {expiry_source}")

    if df_opts.empty:
        print(f"[DEBUG] NO OPTION DATA: {entry_time}, Expiry: {expiry.date()}, "
              f"Type: {opt_type}, Data source: {expiry_source}")
        return None

    best_row = None
    best_delta_diff = np.inf

    for idx, opt in df_opts.iterrows():
        K = float(opt['StrikePrice'])
        option_price = float(opt['Open'])
        T = calculate_time_to_expiry(entry_time.strftime("%Y-%m-%d %H:%M:%S"), expiry.strftime("%d-%m-%y"))
        option_type_bs = 'put' if opt_type == 'PE' else 'call'

        # --- IV calculation ---
        iv = implied_volatility(option_price, spot, K, T, r, option_type_bs)
        if iv is None or iv <= 0:
            print(f"  [DEBUG] IV fail at strike {K} (price={option_price:.2f}), T={T:.6f}")
            continue

        greeks = black_scholes_greeks(spot, K, T, r, iv, option_type_bs)
        if greeks is None:
            print(f"  [DEBUG] Greeks fail at strike {K} (IV={iv:.4f})")
            continue

        delta = greeks['Delta']
        delta_diff = abs(delta - target_delta)
        print(f"  [DEBUG] Strike {K}, Delta={delta:.5f}, IV={iv:.4f}, Δ={delta_diff:.5f}")

        if delta_diff < best_delta_diff:
            best_delta_diff = delta_diff
            best_row = {
                'Entry Time': entry_time,
                'Side': side,
                'Strike': K,
                'Expiry': expiry,
                'IV': iv,
                'Delta': delta,
                'Option Type': opt_type,
                'Option Price': option_price
            }
    if best_row is None:
        print(f"[DEBUG] No strike within delta range for {entry_time} {side}, expiry={expiry.date()}")
    return best_row

import pandas as pd
from typing import Callable, Dict, Optional

def finding_strikes_yearwise(
    trade_log_df: pd.DataFrame,
    find_best_strike_for_signal: Callable[[pd.Series, pd.DataFrame, pd.DataFrame], Optional[dict]],
    nearest_dict: Optional[Dict[int, pd.DataFrame]] = None,
    next_dict: Optional[Dict[int, pd.DataFrame]] = None,
    use_globals: bool = True,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Year-wise wrapper that CALLS your existing `find_best_strike_for_signal(row, options_df, options_df_2)`
    WITHOUT changing any of its logic.

    It:
      1) Detects unique years from trade_log_df['Entry Time']
      2) For each year, fetches the appropriate options data (from provided dicts or from globals())
      3) Iterates trades for that year and calls `find_best_strike_for_signal` per row
      4) Collects non-None results into a single DataFrame and returns it

    Parameters
    ----------
    trade_log_df : pd.DataFrame
        Must include 'Entry Time' column (datetime or parseable).
    find_best_strike_for_signal : Callable
        Your existing function with signature (row, options_df, options_df_2) -> dict | None.
    nearest_dict : dict[int, DataFrame], optional
        Map {year -> nearest-expiry options_df}. If None and use_globals=True, will read from globals().
    next_dict : dict[int, DataFrame], optional
        Map {year -> next-expiry options_df_2}. If None and use_globals=True, will read from globals().
    use_globals : bool
        If True, falls back to globals()[f"options_df_{year}"] and globals()[f"options_df_2_{year}"] when dicts not provided.
    verbose : bool
        Print progress messages.

    Returns
    -------
    pd.DataFrame
        Combined results from all years. Empty DataFrame if nothing returned.
    """
    # --- Validate input ---
    if "Entry Time" not in trade_log_df.columns:
        raise ValueError("trade_log_df must contain 'Entry Time' column.")

    # Ensure datetime
    tl = trade_log_df.copy()
    tl["Entry Time"] = pd.to_datetime(tl["Entry Time"], errors="coerce")
    tl = tl.dropna(subset=["Entry Time"])

    years_in_trades = tl["Entry Time"].dt.year.unique()
    all_results = []

    for year in sorted(years_in_trades):
        if verbose:
            print(f"Processing trades for year: {year}")

        trades_this_year = tl[tl["Entry Time"].dt.year == year]
        if trades_this_year.empty:
            if verbose:
                print(f"  No trades for {year}")
            continue

        # Resolve options data sources
        options_df = None
        options_df_2 = None

        if nearest_dict is not None and next_dict is not None:
            options_df = nearest_dict.get(year, None)
            options_df_2 = next_dict.get(year, None)

        if (options_df is None or options_df_2 is None) and use_globals:
            options_df = options_df or globals().get(f"options_df_{year}")
            options_df_2 = options_df_2 or globals().get(f"options_df_2_{year}")

        if options_df is None or options_df_2 is None:
            if verbose:
                print(f"  Option data missing for {year}")
            continue

        # Call your main function for each trade row
        year_results = []
        for _, row in trades_this_year.iterrows():
            res = find_best_strike_for_signal(row, options_df, options_df_2)
            if res is not None:
                year_results.append(res)

        if year_results:
            all_results.extend(year_results)

    strikes_df = pd.DataFrame(all_results) if all_results else pd.DataFrame()
    if verbose:
        print(f"\nFinished. Rows in strikes_df: {len(strikes_df)}")
    return strikes_df



In [9]:
#OG
csv_path = "/home/newberry3/main/_NIFTY_IDX__202507041318.csv"
spot_df = load_data(csv_path, save_parquet=False, preview=True)
spot_60min = make_spot_60min(spot_df, preview=True)
signals_df = og_signals_generator(spot_60min, preview=True)
trade_log_df = making_tradelogs(spot_60min, signals_df, preview=True)
years = [2021, 2022, 2023, 2024, 2025]
nearest_dict, next_dict, merged_dict = loading_option_data(years)
strikes_df = finding_strikes_yearwise(
    trade_log_df=trade_log_df,
    find_best_strike_for_signal=find_best_strike_for_signal,  # your existing function
    nearest_dict=nearest_dict,  # or None to use globals()
    next_dict=next_dict,        # or None to use globals()
    use_globals=True,
    verbose=True
)
print(strikes_df.head())

spot_df preview:
             Datetime      Open      High       Low     Close
0 2024-05-28 13:57:00  22933.20  22934.35  22926.00  22928.75
1 2024-05-28 13:58:00  22929.05  22933.15  22928.15  22929.60
2 2024-05-28 13:59:00  22929.85  22933.40  22927.50  22929.05
3 2024-05-28 14:00:00  22929.20  22929.55  22921.40  22921.90
4 2024-05-28 14:01:00  22921.50  22931.85  22917.85  22928.20
                Datetime      Open      High       Low     Close       EMA_200
6973 2025-06-13 11:15:00  24652.75  24702.10  24639.75  24688.95  24673.173535
6974 2025-06-13 12:15:00  24688.80  24724.60  24674.45  24686.00  24673.301161
6975 2025-06-13 13:15:00  24685.20  24754.35  24680.65  24715.85  24673.724533
6976 2025-06-13 14:15:00  24714.35  24728.45  24650.90  24725.60  24674.240707
6977 2025-06-13 15:15:00  24725.15  24725.35  24709.75  24711.70  24674.613436
              Datetime           Signal Direction
0  2021-06-09 13:15:00    Bullish_Entry   Bullish
1  2021-06-09 15:15:00  Bullish_Reent

/tmp/ipykernel_144016/3299345248.py:58: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  options_df = pd.concat(dfs, ignore_index=True)


✅ Saved NEAREST expiry options for 2021 to ./options_data_yearwise/options_nearest_2021.parquet
✅ Saved NEXT expiry options for 2021 to ./options_data_yearwise/options_next_2021.parquet

--- Loading option data for year: 2022 ---
✅ Saved NEAREST expiry options for 2022 to ./options_data_yearwise/options_nearest_2022.parquet
✅ Saved NEXT expiry options for 2022 to ./options_data_yearwise/options_next_2022.parquet

--- Loading option data for year: 2023 ---
✅ Saved NEAREST expiry options for 2023 to ./options_data_yearwise/options_nearest_2023.parquet
✅ Saved NEXT expiry options for 2023 to ./options_data_yearwise/options_next_2023.parquet

--- Loading option data for year: 2024 ---
✅ Saved NEAREST expiry options for 2024 to ./options_data_yearwise/options_nearest_2024.parquet
✅ Saved NEXT expiry options for 2024 to ./options_data_yearwise/options_next_2024.parquet

--- Loading option data for year: 2025 ---
✅ Saved NEAREST expiry options for 2025 to ./options_data_yearwise/options_neare

KeyboardInterrupt: 

In [ ]:
strikes_df

,Entry Time,Side,Strike,Expiry,IV,Delta,Option Type,Option Price
0,2021-06-09 13:15:00,Bullish,15600.0,2021-06-17,0.150755,-0.40037,PE,97.05
1,2021-06-09 15:15:00,Bullish,15600.0,2021-06-17,0.147286,-0.42216,PE,100.05
2,2021-06-14 10:15:00,Bullish,15650.0,2021-06-24,0.153836,-0.38562,PE,109.25
3,2021-06-16 10:15:00,Bullish,15750.0,2021-06-24,0.152533,-0.38658,PE,97.40
4,2021-06-16 11:15:00,Bullish,15700.0,2021-06-24,0.151139,-0.37451,PE,91.00
...,...,...,...,...,...,...,...,...
930,2025-06-12 10:15:00,Bullish,25000.0,2025-06-19,0.129466,-0.38305,PE,120.50
931,2025-06-12 11:15:00,Bullish,24950.0,2025-06-19,0.131353,-0.38443,PE,121.20
932,2025-06-12 12:15:00,Bullish,24950.0,2025-06-19,0.131703,-0.38972,PE,122.40
933,2025-06-12 14:15:00,Bullish,24750.0,2025-06-19,0.139597,-0.38665,PE,124.15


In [ ]:
#mod 1 Reentry

In [ ]:
#mod 2 hedge with long put
#changes in find strike function


# MOD 2 (updated): Hedge SHORT PUT with 0.2 LONG PUT; Hedge SHORT CALL with 0.2 LONG CALL
def find_best_strike_for_signal_hedged(row, options_df, options_df_2, r=0.066):
    entry_time = pd.to_datetime(row['Entry Time'])
    spot = float(row['Entry Price'])
    side = row['Side']

    # Primary (existing rule): short ~0.40 |Δ|
    primary_type = 'PE' if side == 'Bullish' else 'CE'
    primary_target_delta = -0.4 if primary_type == 'PE' else 0.4

    # HEDGE (same type as primary): long ~0.20 |Δ|
    hedge_type = primary_type
    hedge_target_delta = -0.2 if hedge_type == 'PE' else 0.2

    # --- Pick correct DataFrame and Expiry ---
    weekday = entry_time.weekday()  # 0=Mon ... 4=Fri
    if weekday == 4:
        df = options_df
        expiry_list = df.loc[df['DateTime'] >= entry_time, 'ExpiryDate'].drop_duplicates().sort_values()
        expiry_source = 'NEAREST (Friday)'
    else:
        df = options_df_2
        expiry_list = df.loc[df['DateTime'] >= entry_time, 'ExpiryDate'].drop_duplicates().sort_values()
        expiry_source = 'NEXT (Mon-Thu)'

    # Ensure correct types
    if df is None or len(df) == 0:
        print(f"[DEBUG] No DF for {entry_time} side={side}")
        return []
    df = df.copy()
    df['StrikePrice'] = df['StrikePrice'].astype(float)
    df['ExpiryDate'] = pd.to_datetime(df['ExpiryDate'])
    df['DateTime'] = pd.to_datetime(df['DateTime'])
    df['Type'] = df['Type'].astype(str)

    # Select expiry
    if expiry_list.empty:
        print(f"[DEBUG] No expiry found for {entry_time} ({side}), source={expiry_source}")
        return []

    expiry = pd.to_datetime(expiry_list.iloc[0])

    # Time window: ±1 minute, combined with same-minute filter
    time_window = pd.Timedelta(minutes=1)
    time_diff = (df['DateTime'] - entry_time).abs()

    def pick_contract(target_type, target_delta):
        # Filter rows to same minute, same expiry, same type, and within ±1 min
        mask = (
            (df['DateTime'].dt.floor('min') == entry_time.floor('min')) &
            (df['ExpiryDate'] == expiry) &
            (df['Type'] == target_type) &
            (time_diff <= time_window)
        )
        df_opts = df[mask].copy()
        print(f"[DEBUG] Entry: {entry_time}, Expiry used: {expiry.date()}, "
              f"Type: {target_type}, Rows in window: {len(df_opts)}, Source: {expiry_source}")

        if df_opts.empty:
            print(f"[DEBUG] NO OPTION DATA: {entry_time}, Expiry: {expiry.date()}, "
                  f"Type: {target_type}, Data source: {expiry_source}")
            return None

        best = None
        best_delta_diff = np.inf

        for _, opt in df_opts.iterrows():
            K = float(opt['StrikePrice'])
            option_price = float(opt['Open'])
            # Helpers you already have:
            T = calculate_time_to_expiry(entry_time.strftime("%Y-%m-%d %H:%M:%S"),
                                         expiry.strftime("%d-%m-%y"))
            option_type_bs = 'put' if target_type == 'PE' else 'call'

            iv = implied_volatility(option_price, spot, K, T, r, option_type_bs)
            if iv is None or iv <= 0:
                continue

            greeks = black_scholes_greeks(spot, K, T, r, iv, option_type_bs)
            if greeks is None:
                continue

            delta = float(greeks['Delta'])
            delta_diff = abs(delta - target_delta)

            if delta_diff < best_delta_diff:
                best_delta_diff = delta_diff
                best = {
                    'Entry Time': entry_time,
                    'Side': side,
                    'Strike': K,
                    'Expiry': expiry,
                    'IV': float(iv),
                    'Delta': delta,
                    'Option Type': target_type,
                    'Option Price': option_price
                }
        return best

    primary_row = pick_contract(primary_type, primary_target_delta)
    hedge_row = pick_contract(hedge_type, hedge_target_delta)

    out = []
    if primary_row is not None:
        primary_row['Hedge'] = False   # primary (short ~0.40 |Δ|)
        out.append(primary_row)
    if hedge_row is not None:
        hedge_row['Hedge'] = True      # hedge (long ~0.20 |Δ| of SAME type)
        out.append(hedge_row)

    if not out:
        print(f"[DEBUG] No strike found for primary/hedge at {entry_time} ({side}), expiry={expiry.date()}")
    return out


# ===== YEARWISE DRIVER (unchanged flow; now picks same-type 0.2Δ hedge) =====

csv_path = "/home/newberry3/main/_NIFTY_IDX__202507041318.csv"
spot_df = load_data(csv_path, save_parquet=False, preview=True)
spot_60min = make_spot_60min(spot_df, preview=True)
signals_df = og_signals_generator(spot_60min, preview=True)
trade_log_df = making_tradelogs(spot_60min, signals_df, preview=True)
years = [2021, 2022, 2023, 2024, 2025]
nearest_dict, next_dict, merged_dict = loading_option_data(years)strikes_df = finding_strikes_yearwise(
    trade_log_df=trade_log_df,
    find_best_strike_for_signal=find_best_strike_for_signal_hedged,  # your existing function
    nearest_dict=nearest_dict,  # or None to use globals()
    next_dict=next_dict,        # or None to use globals()
    use_globals=True,
    verbose=True
)
print(strikes_df.head())